In [1]:
# ============================================================
# CELL 1 — CONFIG + HELPERS
# ============================================================

from pathlib import Path
import time
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# PATH CONFIG
# ============================================================

LAB_DIR = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_DIR = LAB_DIR / "mcs_output"
NEW_DIR = LAB_DIR / "mcs_output_softtest"

RAW_IBTRACS = LAB_DIR / "ibtracs_genesis.csv"
OUT_GENESIS = LAB_DIR / "ibtracs_genesis_firstrecord_2009_2025.csv"

OLD_MCS_CANDIDATES = [
    OLD_DIR / "mcs_tracks_3h_2009_2025_JJASO.csv",
]

NEW_MCS_CANDIDATES = [
    NEW_DIR / "mcs_tracks_soft_3h_2009_2025_JJASO.csv",
    NEW_DIR / "mcs_master_soft_base_2009_2025_JJASO.csv",
]

OLD_LABEL_UPDATED = OLD_DIR / "mcs_labeled_final_old_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv"
NEW_LABEL_UPDATED = NEW_DIR / "mcs_labeled_final_new_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv"

OLD_FEATURE_CANDIDATES = [
    OLD_DIR / "mcs_features_PI_GPI_old_2009_2025_JJASO_zhang4pt.csv",
    OLD_DIR / "mcs_features_PI_GPI_old_2009_2025_JJASO.csv",
    OLD_DIR / "mcs_features_PI_GPI_2009_2025_JJASO_zhang4pt.csv",
    OLD_DIR / "mcs_features_PI_GPI_2009_2025_JJASO_zhang4pt_motionfixed.csv",
]

NEW_FEATURE_CANDIDATES = [
    NEW_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv",
    NEW_DIR / "mcs_features_PI_GPI_new_2009_2025_JJASO_zhang4pt.csv",
]

OLD_MASTER_UPDATED = OLD_DIR / "mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv"
NEW_MASTER_UPDATED = NEW_DIR / "mcs_master_new_exactlead_ready_2009_2025_UPDATED.csv"

OLD_CASE_DIR = OLD_DIR / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
NEW_CASE_DIR = NEW_DIR / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"

OLD_ML_OUT = OLD_DIR / "ml_old_box2deg_2009_2025_UPDATED"
NEW_ML_OUT = NEW_DIR / "ml_new_box2deg_2009_2025_UPDATED"

# ============================================================
# SETTINGS
# ============================================================

LEADS = [6, 12, 24, 48]
MONTHS = [6, 7, 8, 9, 10]

BOX_DEG = 2.0
RADIUS_KM = 300.0
TIME_TOL_H = 1.5

NEG_MULTIPLIER = 5
RANDOM_STATE = 42

ML_FEATURE_COLS = [
    "lon", "lat", "month",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg",
    "eta850_abs", "rh600", "wshear_200_850",
    "pi_vmax_ms", "gpi"
]

ID_COLS = [
    "track_id", "time",
    "centroid_lon", "centroid_lat"
]

BASE_COLS = [
    "track_id", "time",
    "centroid_lon", "centroid_lat",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg"
]

ENV_FEATURE_COLS = [
    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi"
]

OLD_LABEL_COLS = [
    "zhang_y_6h", "zhang_y_12h", "zhang_y_24h", "zhang_y_48h",
    "sens300_y_6h", "sens300_y_12h", "sens300_y_24h", "sens300_y_48h",
]

NEW_LABEL_COLS = [
    "box2deg_y_6h", "box2deg_y_12h", "box2deg_y_24h", "box2deg_y_48h",
    "radius300km_y_6h", "radius300km_y_12h", "radius300km_y_24h", "radius300km_y_48h",
]

MASTER_COLS = [
    "track_id", "time", "month", "lon", "lat",
    "centroid_lon", "centroid_lat",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg",

    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi",

    "box2deg_y_6h", "box2deg_y_12h", "box2deg_y_24h", "box2deg_y_48h",
    "radius300km_y_6h", "radius300km_y_12h", "radius300km_y_24h", "radius300km_y_48h",
]

# ============================================================
# HELPERS
# ============================================================

def find_existing_path(candidates, name, required=False):
    for fp in candidates:
        if fp.exists():
            print(f"[FOUND {name}] {fp}")
            return fp

    print(f"[MISSING {name}] Checked:")
    for fp in candidates:
        print(" ", fp)

    if required:
        raise FileNotFoundError(f"Missing required path for {name}")

    return None


def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def standardize_time(df, col):
    return (
        pd.to_datetime(df[col], errors="coerce", utc=True)
        .dt.tz_convert(None)
    )


def lon_diff_deg(lon_a, lon_b):
    d = (lon_a - lon_b + 180.0) % 360.0 - 180.0
    return np.abs(d)


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c


OLD_MCS_CSV = find_existing_path(OLD_MCS_CANDIDATES, "OLD MCS", required=True)
NEW_MCS_CSV = find_existing_path(NEW_MCS_CANDIDATES, "NEW MCS", required=True)

[FOUND OLD MCS] C:\Users\_s2218026\Documents\lab\mcs_output\mcs_tracks_3h_2009_2025_JJASO.csv
[FOUND NEW MCS] C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_tracks_soft_3h_2009_2025_JJASO.csv


In [2]:
# ============================================================
# CELL 2 — BUILD CORRECT CANONICAL IBTRACS GENESIS FILE
# ============================================================

print("\n" + "=" * 90)
print("BUILD CORRECT CANONICAL IBTRACS GENESIS FILE")
print("=" * 90)

if not RAW_IBTRACS.exists():
    raise FileNotFoundError(RAW_IBTRACS)

raw = pd.read_csv(RAW_IBTRACS, low_memory=False)

sid_col = find_col(raw, ["SID", "sid", "storm_id", "id"])
time_col = find_col(raw, ["genesis_time", "ISO_TIME", "iso_time", "time", "datetime"])
lat_col = find_col(raw, ["genesis_lat", "LAT", "lat", "latitude", "USA_LAT", "usa_lat"])
lon_col = find_col(raw, ["genesis_lon", "LON", "lon", "longitude", "USA_LON", "usa_lon"])

print("Detected columns:")
print("sid_col :", sid_col)
print("time_col:", time_col)
print("lat_col :", lat_col)
print("lon_col :", lon_col)

if sid_col is None or time_col is None or lat_col is None or lon_col is None:
    raise ValueError("Could not detect SID/time/lat/lon columns in ibtracs_genesis.csv")

raw["time_tmp"] = standardize_time(raw, time_col)
raw["lat_tmp"] = pd.to_numeric(raw[lat_col], errors="coerce")
raw["lon_tmp"] = pd.to_numeric(raw[lon_col], errors="coerce") % 360.0

raw = raw.dropna(subset=[sid_col, "time_tmp", "lat_tmp", "lon_tmp"]).copy()

first = (
    raw.sort_values([sid_col, "time_tmp"])
    .groupby(sid_col, as_index=False)
    .first()
)

gen = pd.DataFrame({
    "SID": first[sid_col],
    "genesis_time": first["time_tmp"],
    "genesis_lat": first["lat_tmp"],
    "genesis_lon": first["lon_tmp"],
})

for c in ["SEASON", "BASIN", "SUBBASIN", "NAME", "NATURE"]:
    if c in first.columns:
        gen[c] = first[c]

gen = gen[
    gen["genesis_time"].dt.year.between(2009, 2025)
].copy()

gen = gen.sort_values("genesis_time").reset_index(drop=True)

gen.to_csv(OUT_GENESIS, index=False)

print("\n[SAVED GENESIS]")
print(OUT_GENESIS)
print("Rows:", len(gen))
print("Time range:", gen["genesis_time"].min(), "to", gen["genesis_time"].max())

print("\nGenesis counts by year:")
print(gen["genesis_time"].dt.year.value_counts().sort_index())

wnp_jjaso = gen[
    gen["genesis_time"].dt.year.between(2009, 2025)
    & gen["genesis_time"].dt.month.isin(MONTHS)
    & gen["genesis_lat"].between(-2, 32)
    & gen["genesis_lon"].between(98, 182)
].copy()

print("\nWNP JJASO genesis counts by year:")
print(wnp_jjaso["genesis_time"].dt.year.value_counts().sort_index())

print("\n2025 WNP JJASO genesis events:")
display(wnp_jjaso[wnp_jjaso["genesis_time"].dt.year == 2025])


BUILD CORRECT CANONICAL IBTRACS GENESIS FILE
Detected columns:
sid_col : SID
time_col: ISO_TIME
lat_col : LAT
lon_col : LON


C:\Users\_s2218026\AppData\Local\Temp\ipykernel_1001124\3094022822.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(df[col], errors="coerce", utc=True)



[SAVED GENESIS]
C:\Users\_s2218026\Documents\lab\ibtracs_genesis_firstrecord_2009_2025.csv
Rows: 1741
Time range: 2009-01-07 12:00:00 to 2025-11-04 06:00:00

Genesis counts by year:
genesis_time
2009    100
2010     88
2011     95
2012     94
2013    102
2014     89
2015    112
2016     96
2017    108
2018    117
2019    109
2020    121
2021    106
2022    113
2023     88
2024    104
2025     99
Name: count, dtype: int64

WNP JJASO genesis counts by year:
genesis_time
2009    23
2010    17
2011    19
2012    20
2013    28
2014    16
2015    19
2016    26
2017    26
2018    27
2019    22
2020    24
2021    21
2022    26
2023    14
2024    22
2025    29
Name: count, dtype: int64

2025 WNP JJASO genesis events:


C:\Users\_s2218026\AppData\Local\Temp\ipykernel_1001124\1997362873.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .first()


,SID,genesis_time,genesis_lat,genesis_lon,SEASON,BASIN,SUBBASIN,NAME,NATURE
1672,2025162N15114,2025-06-10 12:00:00,15.0,113.9,2025,WP,MM,WUTIP,NR
1676,2025174N23146,2025-06-22 12:00:00,23.2,145.9,2025,WP,MM,SEPAT,NR
1678,2025176N18113,2025-06-25 06:00:00,18.5,113.2,2025,WP,MM,UNNAMED,NR
1680,2025183N24148,2025-07-01 18:00:00,24.2,147.6,2025,WP,MM,MUN,NR
1682,2025185N20119,2025-07-04 00:00:00,19.7,118.7,2025,WP,MM,DANAS,NR
1683,2025192N25140,2025-07-11 00:00:00,25.0,140.4,2025,WP,MM,NARI,NR
1684,2025194N31124,2025-07-12 18:00:00,30.8,123.6,2025,WP,MM,UNNAMED,NR
1687,2025200N19123,2025-07-18 12:00:00,19.3,122.5,2025,WP,MM,WIPHA,NR
1688,2025203N18133,2025-07-22 06:00:00,17.9,132.5,2025,WP,MM,FRANCISCO,NR
1689,2025204N19121,2025-07-23 00:00:00,18.8,120.5,2025,WP,MM,CO-MAY,NR


In [3]:
# ============================================================
# CELL 3 — DEFINE EXACT-LEAD RELABELING FUNCTION
# ============================================================

def label_mcs_exactlead(mcs_csv, genesis_csv, out_csv, dataset_name):
    print("\n" + "=" * 90)
    print(f"RELABEL {dataset_name.upper()}")
    print("=" * 90)

    mcs_csv = Path(mcs_csv)
    genesis_csv = Path(genesis_csv)
    out_csv = Path(out_csv)

    if not mcs_csv.exists():
        raise FileNotFoundError(mcs_csv)
    if not genesis_csv.exists():
        raise FileNotFoundError(genesis_csv)

    mcs = pd.read_csv(mcs_csv, low_memory=False)
    gen = pd.read_csv(genesis_csv, low_memory=False)

    mcs["time"] = standardize_time(mcs, "time")
    gen["genesis_time"] = standardize_time(gen, "genesis_time")

    mcs["centroid_lat"] = pd.to_numeric(mcs["centroid_lat"], errors="coerce")
    mcs["centroid_lon"] = pd.to_numeric(mcs["centroid_lon"], errors="coerce") % 360.0

    gen["genesis_lat"] = pd.to_numeric(gen["genesis_lat"], errors="coerce")
    gen["genesis_lon"] = pd.to_numeric(gen["genesis_lon"], errors="coerce") % 360.0

    mcs = mcs.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()
    gen = gen.dropna(subset=["genesis_time", "genesis_lat", "genesis_lon"]).copy()

    mcs = mcs[
        mcs["time"].dt.year.between(2009, 2025)
        & mcs["time"].dt.month.isin(MONTHS)
    ].copy()

    gen = gen[
        gen["genesis_time"].dt.year.between(2009, 2025)
        & gen["genesis_time"].dt.month.isin(MONTHS)
        & gen["genesis_lat"].between(-2, 32)
        & gen["genesis_lon"].between(98, 182)
    ].copy()

    gen = gen[
        (gen["genesis_time"] >= mcs["time"].min())
        & (gen["genesis_time"] <= mcs["time"].max())
    ].copy()

    mcs = mcs.sort_values("time").reset_index(drop=True)
    gen = gen.sort_values("genesis_time").reset_index(drop=True)

    print("MCS rows:", len(mcs))
    print("Genesis rows after filtering:", len(gen))
    print("MCS time range:", mcs["time"].min(), "to", mcs["time"].max())
    print("Genesis time range:", gen["genesis_time"].min(), "to", gen["genesis_time"].max())

    print("\nGenesis counts by year:")
    print(gen["genesis_time"].dt.year.value_counts().sort_index())

    label_like = []

    for L in LEADS:
        label_like += [
            f"zhang_y_{L}h", f"sens300_y_{L}h",
            f"box2deg_y_{L}h", f"radius300km_y_{L}h",
            f"zhang_match_count_{L}h", f"sens300_match_count_{L}h",
        ]

    mcs = mcs.drop(columns=[c for c in label_like if c in mcs.columns], errors="ignore")

    for L in LEADS:
        mcs[f"zhang_y_{L}h"] = 0
        mcs[f"sens300_y_{L}h"] = 0
        mcs[f"zhang_match_count_{L}h"] = 0
        mcs[f"sens300_match_count_{L}h"] = 0

    t0 = time.time()

    for gi, g in gen.iterrows():
        gtime = g["genesis_time"]
        glat = float(g["genesis_lat"])
        glon = float(g["genesis_lon"])

        for L in LEADS:
            target_time = gtime - pd.Timedelta(hours=L)

            cand = mcs[
                (mcs["time"] >= target_time - pd.Timedelta(hours=TIME_TOL_H))
                & (mcs["time"] <= target_time + pd.Timedelta(hours=TIME_TOL_H))
            ]

            if cand.empty:
                continue

            dlat = np.abs(cand["centroid_lat"].values - glat)
            dlon = lon_diff_deg(cand["centroid_lon"].values, glon)
            in_box = (dlat <= BOX_DEG) & (dlon <= BOX_DEG)

            dkm = haversine_km(
                cand["centroid_lat"].values,
                cand["centroid_lon"].values,
                np.full(len(cand), glat),
                np.full(len(cand), glon)
            )

            in_300 = dkm <= RADIUS_KM

            box_idx = cand.index[in_box]
            km_idx = cand.index[in_300]

            if len(box_idx) > 0:
                mcs.loc[box_idx, f"zhang_y_{L}h"] = 1
                mcs.loc[box_idx, f"zhang_match_count_{L}h"] += 1

            if len(km_idx) > 0:
                mcs.loc[km_idx, f"sens300_y_{L}h"] = 1
                mcs.loc[km_idx, f"sens300_match_count_{L}h"] += 1

        if (gi + 1) % 20 == 0 or (gi + 1) == len(gen):
            elapsed = (time.time() - t0) / 60
            print(f"Genesis events processed: {gi + 1}/{len(gen)} | elapsed {elapsed:.1f} min")

    for L in LEADS:
        mcs[f"box2deg_y_{L}h"] = mcs[f"zhang_y_{L}h"].astype(int)
        mcs[f"radius300km_y_{L}h"] = mcs[f"sens300_y_{L}h"].astype(int)

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    mcs.to_csv(out_csv, index=False)

    print("\n[SAVED LABEL FILE]")
    print(out_csv)

    print("\nPositive counts:")
    for L in LEADS:
        print(
            f"{L}h | "
            f"box2deg={int(mcs[f'box2deg_y_{L}h'].sum())} | "
            f"radius300km={int(mcs[f'radius300km_y_{L}h'].sum())}"
        )

    rows = []

    for y, gy in mcs.groupby(mcs["time"].dt.year):
        row = {"year": int(y), "rows": len(gy)}

        for L in LEADS:
            row[f"box2deg_{L}h"] = int(gy[f"box2deg_y_{L}h"].sum())
            row[f"radius300km_{L}h"] = int(gy[f"radius300km_y_{L}h"].sum())

        rows.append(row)

    qc = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)

    qc_out = out_csv.with_name(out_csv.stem + "_year_qc.csv")
    qc.to_csv(qc_out, index=False)

    print("\nYear QC:")
    display(qc)

    print("\n[SAVED YEAR QC]")
    print(qc_out)

    return qc

In [4]:
# ============================================================
# CELL 4 — RUN RELABELING FOR OLD AND NEW
# ============================================================

old_qc = label_mcs_exactlead(
    mcs_csv=OLD_MCS_CSV,
    genesis_csv=OUT_GENESIS,
    out_csv=OLD_LABEL_UPDATED,
    dataset_name="old"
)

new_qc = label_mcs_exactlead(
    mcs_csv=NEW_MCS_CSV,
    genesis_csv=OUT_GENESIS,
    out_csv=NEW_LABEL_UPDATED,
    dataset_name="new"
)

print("\nOLD 2025 QC:")
display(old_qc[old_qc["year"] == 2025])

print("\nNEW 2025 QC:")
display(new_qc[new_qc["year"] == 2025])


RELABEL OLD
MCS rows: 900585
Genesis rows after filtering: 379
MCS time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Genesis time range: 2009-06-13 06:00:00 to 2025-10-18 00:00:00

Genesis counts by year:
genesis_time
2009    23
2010    17
2011    19
2012    20
2013    28
2014    16
2015    19
2016    26
2017    26
2018    27
2019    22
2020    24
2021    21
2022    26
2023    14
2024    22
2025    29
Name: count, dtype: int64
Genesis events processed: 20/379 | elapsed 0.0 min
Genesis events processed: 40/379 | elapsed 0.0 min
Genesis events processed: 60/379 | elapsed 0.0 min
Genesis events processed: 80/379 | elapsed 0.0 min
Genesis events processed: 100/379 | elapsed 0.0 min
Genesis events processed: 120/379 | elapsed 0.0 min
Genesis events processed: 140/379 | elapsed 0.0 min
Genesis events processed: 160/379 | elapsed 0.0 min
Genesis events processed: 180/379 | elapsed 0.1 min
Genesis events processed: 200/379 | elapsed 0.1 min
Genesis events processed: 220/379 | elapsed 0.1

,year,rows,box2deg_6h,radius300km_6h,box2deg_12h,radius300km_12h,box2deg_24h,radius300km_24h,box2deg_48h,radius300km_48h
0,2009,44437,14,17,10,14,7,8,10,12
1,2010,47415,10,14,4,8,5,5,6,8
2,2011,47582,4,9,8,10,6,9,2,6
3,2012,46154,4,4,4,5,6,7,4,6
4,2013,46165,12,20,11,16,8,12,8,10
5,2014,45964,7,9,8,8,5,5,2,4
6,2015,37512,11,18,9,14,8,10,5,7
7,2016,58611,22,29,14,19,15,22,16,26
8,2017,59282,24,34,24,29,17,22,8,8
9,2018,55511,25,32,16,25,17,23,14,20



[SAVED YEAR QC]
C:\Users\_s2218026\Documents\lab\mcs_output\mcs_labeled_final_old_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED_year_qc.csv

RELABEL NEW
MCS rows: 718030
Genesis rows after filtering: 379
MCS time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Genesis time range: 2009-06-13 06:00:00 to 2025-10-18 00:00:00

Genesis counts by year:
genesis_time
2009    23
2010    17
2011    19
2012    20
2013    28
2014    16
2015    19
2016    26
2017    26
2018    27
2019    22
2020    24
2021    21
2022    26
2023    14
2024    22
2025    29
Name: count, dtype: int64
Genesis events processed: 20/379 | elapsed 0.0 min
Genesis events processed: 40/379 | elapsed 0.0 min
Genesis events processed: 60/379 | elapsed 0.0 min
Genesis events processed: 80/379 | elapsed 0.0 min
Genesis events processed: 100/379 | elapsed 0.0 min
Genesis events processed: 120/379 | elapsed 0.0 min
Genesis events processed: 140/379 | elapsed 0.0 min
Genesis events processed: 160/379 | elapsed 0.0 min
Ge

,year,rows,box2deg_6h,radius300km_6h,box2deg_12h,radius300km_12h,box2deg_24h,radius300km_24h,box2deg_48h,radius300km_48h
0,2009,36527,13,16,9,12,5,7,7,10
1,2010,38300,8,10,4,8,2,3,5,7
2,2011,39016,4,8,8,10,5,8,2,4
3,2012,38273,4,4,3,3,2,3,3,6
4,2013,38101,10,16,8,12,7,10,7,10
5,2014,38975,5,6,7,7,5,5,2,3
6,2015,31783,11,14,6,11,8,9,5,6
7,2016,45360,12,14,8,12,8,13,8,11
8,2017,45867,16,19,10,17,8,11,4,6
9,2018,43378,10,15,12,17,8,13,7,14



[SAVED YEAR QC]
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_labeled_final_new_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED_year_qc.csv

OLD 2025 QC:


,year,rows,box2deg_6h,radius300km_6h,box2deg_12h,radius300km_12h,box2deg_24h,radius300km_24h,box2deg_48h,radius300km_48h
16,2025,60874,21,25,21,29,14,20,9,17



NEW 2025 QC:


,year,rows,box2deg_6h,radius300km_6h,box2deg_12h,radius300km_12h,box2deg_24h,radius300km_24h,box2deg_48h,radius300km_48h
16,2025,47944,13,16,11,18,9,12,7,11


In [5]:
# ============================================================
# CELL 5 — DEFINE UPDATED MASTER TABLE BUILDER
# ============================================================

def build_master_table(features_csv, labels_csv, out_master_csv, dataset_name):
    print("\n" + "=" * 90)
    print(f"BUILD UPDATED MASTER TABLE: {dataset_name.upper()}")
    print("=" * 90)

    if features_csv is None or not Path(features_csv).exists():
        print(f"[SKIP MASTER] Missing feature file for {dataset_name}: {features_csv}")
        return None

    features_csv = Path(features_csv)
    labels_csv = Path(labels_csv)
    out_master_csv = Path(out_master_csv)

    if not labels_csv.exists():
        raise FileNotFoundError(labels_csv)

    feat = pd.read_csv(features_csv, low_memory=False)
    lab = pd.read_csv(labels_csv, low_memory=False)

    feat["time"] = standardize_time(feat, "time")
    lab["time"] = standardize_time(lab, "time")

    feat["track_id"] = feat["track_id"].astype(str)
    lab["track_id"] = lab["track_id"].astype(str)

    for c in ["centroid_lon", "centroid_lat"]:
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        lab[c] = pd.to_numeric(lab[c], errors="coerce")

    feat["centroid_lon"] = feat["centroid_lon"] % 360.0
    lab["centroid_lon"] = lab["centroid_lon"] % 360.0

    for L in LEADS:
        if f"box2deg_y_{L}h" not in lab.columns and f"zhang_y_{L}h" in lab.columns:
            lab[f"box2deg_y_{L}h"] = lab[f"zhang_y_{L}h"].astype(int)

        if f"radius300km_y_{L}h" not in lab.columns and f"sens300_y_{L}h" in lab.columns:
            lab[f"radius300km_y_{L}h"] = lab[f"sens300_y_{L}h"].astype(int)

    required_feat = BASE_COLS + ENV_FEATURE_COLS
    required_lab = ["track_id", "time", "centroid_lon", "centroid_lat"] + NEW_LABEL_COLS

    missing_feat = [c for c in required_feat if c not in feat.columns]
    missing_lab = [c for c in required_lab if c not in lab.columns]

    if missing_feat:
        raise ValueError(f"Missing feature columns for {dataset_name}:\n{missing_feat}")

    if missing_lab:
        raise ValueError(f"Missing label columns for {dataset_name}:\n{missing_lab}")

    all_label_cols_possible = OLD_LABEL_COLS + NEW_LABEL_COLS
    all_label_cols_possible += [f"zhang_match_count_{L}h" for L in LEADS]
    all_label_cols_possible += [f"sens300_match_count_{L}h" for L in LEADS]

    feat = feat.drop(
        columns=[c for c in all_label_cols_possible if c in feat.columns],
        errors="ignore"
    )

    merge_keys = ["track_id", "time", "centroid_lon", "centroid_lat"]

    feat = feat.dropna(subset=merge_keys).copy()
    lab = lab.dropna(subset=merge_keys).copy()

    feat = feat.drop_duplicates(subset=merge_keys).copy()
    lab_small = lab[merge_keys + NEW_LABEL_COLS].drop_duplicates(subset=merge_keys).copy()

    print("Feature rows:", len(feat))
    print("Label rows:", len(lab_small))
    print("Feature time range:", feat["time"].min(), "to", feat["time"].max())
    print("Label time range:", lab_small["time"].min(), "to", lab_small["time"].max())

    df = feat.merge(
        lab_small,
        on=merge_keys,
        how="left"
    )

    df["month"] = df["time"].dt.month
    df["lon"] = df["centroid_lon"]
    df["lat"] = df["centroid_lat"]

    for c in NEW_LABEL_COLS:
        n_missing = int(df[c].isna().sum())

        if n_missing > 0:
            raise ValueError(f"{dataset_name}: {c} has {n_missing} missing values after merge.")

        df[c] = df[c].astype(int)

    missing_master = [c for c in MASTER_COLS if c not in df.columns]
    if missing_master:
        raise ValueError(f"{dataset_name}: missing master columns:\n{missing_master}")

    master = (
        df[MASTER_COLS]
        .sort_values(["time", "track_id"])
        .reset_index(drop=True)
    )

    out_master_csv.parent.mkdir(parents=True, exist_ok=True)
    master.to_csv(out_master_csv, index=False)

    print("\n[SAVED MASTER]")
    print(out_master_csv)
    print("Shape:", master.shape)
    print("Time range:", master["time"].min(), "to", master["time"].max())
    print("Years:", master["time"].dt.year.min(), "-", master["time"].dt.year.max())

    print("\nPositive counts:")
    for L in LEADS:
        print(
            f"{L}h | "
            f"box2deg={int(master[f'box2deg_y_{L}h'].sum())} | "
            f"radius300km={int(master[f'radius300km_y_{L}h'].sum())}"
        )

    print("\nNaN fractions in key predictors:")
    print(master[[
        "eta850_abs", "rh600", "wshear_200_850",
        "sst_K", "mslp_Pa", "pi_vmax_ms", "gpi",
        "speed_kmh", "direction_deg"
    ]].isna().mean())

    return master

In [6]:
# ============================================================
# CELL 6 — BUILD UPDATED OLD AND NEW MASTER TABLES
# ============================================================

OLD_FEATURES_CSV = find_existing_path(OLD_FEATURE_CANDIDATES, "OLD FEATURES")
NEW_FEATURES_CSV = find_existing_path(NEW_FEATURE_CANDIDATES, "NEW FEATURES")

old_master = build_master_table(
    features_csv=OLD_FEATURES_CSV,
    labels_csv=OLD_LABEL_UPDATED,
    out_master_csv=OLD_MASTER_UPDATED,
    dataset_name="old"
)

new_master = build_master_table(
    features_csv=NEW_FEATURES_CSV,
    labels_csv=NEW_LABEL_UPDATED,
    out_master_csv=NEW_MASTER_UPDATED,
    dataset_name="new"
)

[FOUND OLD FEATURES] C:\Users\_s2218026\Documents\lab\mcs_output\mcs_features_PI_GPI_old_2009_2025_JJASO_zhang4pt.csv
[FOUND NEW FEATURES] C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv

BUILD UPDATED MASTER TABLE: OLD
Feature rows: 900585
Label rows: 900585
Feature time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Label time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00

[SAVED MASTER]
C:\Users\_s2218026\Documents\lab\mcs_output\mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv
Shape: (900585, 34)
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Years: 2009 - 2025

Positive counts:
6h | box2deg=239 | radius300km=342
12h | box2deg=226 | radius300km=314
24h | box2deg=187 | radius300km=259
48h | box2deg=134 | radius300km=196

NaN fractions in key predictors:
eta850_abs        0.067594
rh600             0.067594
wshear_200_850    0.067594
sst_K             0.225609
mslp_Pa           0.067594
pi_vmax_ms        0.

In [7]:
# ============================================================
# CELL 7 — DEFINE UPDATED CASE TABLE BUILDER
# ============================================================

def build_case_tables_from_master(master_csv, out_case_dir, dataset_name):
    print("\n" + "=" * 90)
    print(f"BUILD CASE TABLES: {dataset_name.upper()}")
    print("=" * 90)

    master_csv = Path(master_csv)
    out_case_dir = Path(out_case_dir)
    out_case_dir.mkdir(parents=True, exist_ok=True)

    if not master_csv.exists():
        print(f"[SKIP CASE TABLES] Missing master file for {dataset_name}: {master_csv}")
        return None

    df = pd.read_csv(master_csv, low_memory=False)
    df["time"] = standardize_time(df, "time")

    required_cols = ID_COLS + ML_FEATURE_COLS + [f"box2deg_y_{L}h" for L in LEADS]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(f"{dataset_name}: missing required columns:\n{missing}")

    print("Master rows:", len(df))
    print("Time range:", df["time"].min(), "to", df["time"].max())

    before = len(df)
    after = len(df)

    print("Rows before predictor NaN drop:", before)
    print("Rows after predictor NaN drop:", after)
    print("Dropped:", before - after)

    any_positive = np.zeros(len(df), dtype=bool)

    for L in LEADS:
        any_positive |= (df[f"box2deg_y_{L}h"].astype(int).values == 1)

    neg_pool = df.loc[~any_positive].copy()

    print("Never-positive negative pool:", len(neg_pool))

    summary_rows = []

    for L in LEADS:
        label_col = f"box2deg_y_{L}h"

        pos = df[df[label_col].astype(int) == 1].copy()
        pos["lead_h"] = L
        pos["y"] = 1

        n_pos = len(pos)

        if n_pos == 0:
            print(f"[WARN] {dataset_name} {L}h has zero positives. Skipping.")
            continue

        n_neg_target = NEG_MULTIPLIER * n_pos

        if len(neg_pool) < n_neg_target:
            print(f"[WARN] Not enough negatives for {dataset_name} {L}h. Using all available negatives.")
            neg = neg_pool.copy()
        else:
            neg = neg_pool.sample(n=n_neg_target, random_state=RANDOM_STATE + L).copy()

        neg["lead_h"] = L
        neg["y"] = 0

        keep_cols = ID_COLS + ML_FEATURE_COLS + ["lead_h", "y"]

        case = pd.concat(
            [pos[keep_cols], neg[keep_cols]],
            ignore_index=True
        )

        case = case.sample(frac=1, random_state=RANDOM_STATE + L).reset_index(drop=True)

        out_csv = out_case_dir / f"box2deg_case_table_{dataset_name}_{L}h.csv"
        case.to_csv(out_csv, index=False)

        n_pos_final = int((case["y"] == 1).sum())
        n_neg_final = int((case["y"] == 0).sum())

        summary_rows.append({
            "dataset": dataset_name,
            "label_family": "box2deg",
            "lead_h": L,
            "positive": n_pos_final,
            "negative": n_neg_final,
            "total": len(case),
            "pos_frac": n_pos_final / len(case),
        })

        print("\n[SAVED]", out_csv.name)
        print("Positive:", n_pos_final)
        print("Negative:", n_neg_final)
        print("Total:", len(case))
        print("Pos frac:", n_pos_final / len(case))

    summary = pd.DataFrame(summary_rows)

    summary_csv = out_case_dir / f"case_table_summary_{dataset_name}_box2deg_2009_2025_UPDATED.csv"
    summary.to_csv(summary_csv, index=False)

    print("\n[SAVED CASE SUMMARY]")
    print(summary_csv)
    display(summary)

    return summary

In [8]:
# ============================================================
# CELL 8 — BUILD UPDATED OLD AND NEW CASE TABLES
# ============================================================

old_case_summary = None

if old_master is not None:
    old_case_summary = build_case_tables_from_master(
        master_csv=OLD_MASTER_UPDATED,
        out_case_dir=OLD_CASE_DIR,
        dataset_name="old"
    )

new_case_summary = None

if new_master is not None:
    new_case_summary = build_case_tables_from_master(
        master_csv=NEW_MASTER_UPDATED,
        out_case_dir=NEW_CASE_DIR,
        dataset_name="new"
    )


BUILD CASE TABLES: OLD
Master rows: 900585
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Rows before predictor NaN drop: 900585
Rows after predictor NaN drop: 900585
Dropped: 0
Never-positive negative pool: 899799

[SAVED] box2deg_case_table_old_6h.csv
Positive: 239
Negative: 1195
Total: 1434
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_old_12h.csv
Positive: 226
Negative: 1130
Total: 1356
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_old_24h.csv
Positive: 187
Negative: 935
Total: 1122
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_old_48h.csv
Positive: 134
Negative: 670
Total: 804
Pos frac: 0.16666666666666666

[SAVED CASE SUMMARY]
C:\Users\_s2218026\Documents\lab\mcs_output\exactlead_box2deg_case_tables_old_2009_2025_UPDATED\case_table_summary_old_box2deg_2009_2025_UPDATED.csv


,dataset,label_family,lead_h,positive,negative,total,pos_frac
0,old,box2deg,6,239,1195,1434,0.166667
1,old,box2deg,12,226,1130,1356,0.166667
2,old,box2deg,24,187,935,1122,0.166667
3,old,box2deg,48,134,670,804,0.166667



BUILD CASE TABLES: NEW
Master rows: 718030
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Rows before predictor NaN drop: 718030
Rows after predictor NaN drop: 718030
Dropped: 0
Never-positive negative pool: 717508

[SAVED] box2deg_case_table_new_6h.csv
Positive: 166
Negative: 830
Total: 996
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_new_12h.csv
Positive: 149
Negative: 745
Total: 894
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_new_24h.csv
Positive: 119
Negative: 595
Total: 714
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_new_48h.csv
Positive: 88
Negative: 440
Total: 528
Pos frac: 0.16666666666666666

[SAVED CASE SUMMARY]
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\exactlead_box2deg_case_tables_new_2009_2025_UPDATED\case_table_summary_new_box2deg_2009_2025_UPDATED.csv


,dataset,label_family,lead_h,positive,negative,total,pos_frac
0,new,box2deg,6,166,830,996,0.166667
1,new,box2deg,12,149,745,894,0.166667
2,new,box2deg,24,119,595,714,0.166667
3,new,box2deg,48,88,440,528,0.166667


In [9]:
# ============================================================
# CELL 9 — PRINT FINAL PATHS FOR ML
# ============================================================

print("\n" + "=" * 90)
print("DONE — USE THESE PATHS FOR ML")
print("=" * 90)

print("\nOLD updated label:")
print(OLD_LABEL_UPDATED)

print("\nNEW updated label:")
print(NEW_LABEL_UPDATED)

print("\nOLD updated master:")
print(OLD_MASTER_UPDATED)

print("\nNEW updated master:")
print(NEW_MASTER_UPDATED)

print("\nOLD updated case dir:")
print(OLD_CASE_DIR)

print("\nNEW updated case dir:")
print(NEW_CASE_DIR)

print("\nFor OLD ML:")
print(f'CASE_DIR = Path(r"{OLD_CASE_DIR}")')
print(f'OUT_DIR  = Path(r"{OLD_ML_OUT}")')

print("\nFor NEW ML:")
print(f'CASE_DIR = Path(r"{NEW_CASE_DIR}")')
print(f'OUT_DIR  = Path(r"{NEW_ML_OUT}")')

if NEW_FEATURES_CSV is None:
    print("\nNOTE:")
    print("NEW/SOFT full 2009–2025 feature file was not found.")
    print("The code relabeled NEW successfully, but skipped NEW master/case tables.")
    print("Finish NEW 2025 feature extraction, concatenate the 2009–2025 feature file, then rerun Cells 6–9.")


DONE — USE THESE PATHS FOR ML

OLD updated label:
C:\Users\_s2218026\Documents\lab\mcs_output\mcs_labeled_final_old_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv

NEW updated label:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_labeled_final_new_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv

OLD updated master:
C:\Users\_s2218026\Documents\lab\mcs_output\mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv

NEW updated master:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_master_new_exactlead_ready_2009_2025_UPDATED.csv

OLD updated case dir:
C:\Users\_s2218026\Documents\lab\mcs_output\exactlead_box2deg_case_tables_old_2009_2025_UPDATED

NEW updated case dir:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\exactlead_box2deg_case_tables_new_2009_2025_UPDATED

For OLD ML:
CASE_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output\exactlead_box2deg_case_tables_old_2009_2025_UPDATED")
OUT_DIR  = Path(r"C:\Users\_s2218026\Documents\lab\

In [13]:
from pathlib import Path
import pandas as pd

case_dirs = {
    "OLD": Path(r"C:\Users\_s2218026\Documents\lab\mcs_output\exactlead_box2deg_case_tables_old_2009_2025_UPDATED"),
    "NEW": Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\exactlead_box2deg_case_tables_new_2009_2025_UPDATED"),
}

for name, case_dir in case_dirs.items():
    print("\n" + "=" * 70)
    print(name)
    print("Directory exists:", case_dir.exists())

    for L in [6, 12, 24, 48]:
        fp = case_dir / f"box2deg_case_table_{name.lower()}_{L}h.csv"

        print(f"\nLead {L}h")
        print("exists:", fp.exists())

        if fp.exists():
            df = pd.read_csv(fp)
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

            print("rows:", len(df))
            print("positive:", int(df["y"].sum()))
            print("negative:", int((df["y"] == 0).sum()))
            print("time range:", df["time"].min(), "to", df["time"].max())
            print("2025 rows:", int((df["time"].dt.year == 2025).sum()))
            print("2025 positives:", int(df.loc[df["time"].dt.year == 2025, "y"].sum()))


OLD
Directory exists: True

Lead 6h
exists: True
rows: 1434
positive: 239
negative: 1195
time range: 2009-06-01 09:00:00 to 2025-10-25 15:00:00
2025 rows: 101
2025 positives: 21

Lead 12h
exists: True
rows: 1356
positive: 226
negative: 1130
time range: 2009-06-01 00:00:00 to 2025-10-30 15:00:00
2025 rows: 91
2025 positives: 21

Lead 24h
exists: True
rows: 1122
positive: 187
negative: 935
time range: 2009-06-03 09:00:00 to 2025-10-27 12:00:00
2025 rows: 84
2025 positives: 14

Lead 48h
exists: True
rows: 804
positive: 134
negative: 670
time range: 2009-06-02 09:00:00 to 2025-10-28 09:00:00
2025 rows: 61
2025 positives: 9

NEW
Directory exists: True

Lead 6h
exists: True
rows: 996
positive: 166
negative: 830
time range: 2009-06-04 18:00:00 to 2025-10-30 15:00:00
2025 rows: 66
2025 positives: 13

Lead 12h
exists: True
rows: 894
positive: 149
negative: 745
time range: 2009-06-05 15:00:00 to 2025-10-27 12:00:00
2025 rows: 53
2025 positives: 11

Lead 24h
exists: True
rows: 714
positive: 119


In [11]:
from pathlib import Path

NEW_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

full_feature = NEW_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

monthly_dir = NEW_DIR / "features_monthly_soft_2009_2024"

print("Full NEW feature file exists:", full_feature.exists())
print(full_feature)

expected = [
    monthly_dir / f"mcs_features_PI_GPI_soft_{y}{m:02d}.csv"
    for y in range(2009, 2026)
    for m in [6, 7, 8, 9, 10]
]

missing = [fp for fp in expected if not fp.exists()]

print("\nExpected monthly files:", len(expected))
print("Missing monthly files:", len(missing))

for fp in missing:
    print(fp.name)

Full NEW feature file exists: True
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv

Expected monthly files: 85
Missing monthly files: 0


In [12]:
# ============================================================
# CHECK OLD 2025 VALIDITY
# Features do not use IBTrACS; labels/master/cases should use updated IBTrACS.
# ============================================================

from pathlib import Path
import pandas as pd

OLD_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output")

OLD_FEATURE_CANDIDATES = [
    OLD_DIR / "mcs_features_PI_GPI_old_2009_2025_JJASO_zhang4pt.csv",
    OLD_DIR / "mcs_features_PI_GPI_old_2009_2025_JJASO.csv",
    OLD_DIR / "mcs_features_PI_GPI_2009_2025_JJASO_zhang4pt.csv",
    OLD_DIR / "mcs_features_PI_GPI_2009_2025_JJASO_zhang4pt_motionfixed.csv",
]

OLD_LABEL_UPDATED = OLD_DIR / "mcs_labeled_final_old_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv"
OLD_MASTER_UPDATED = OLD_DIR / "mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv"
OLD_CASE_DIR = OLD_DIR / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"

# ------------------------------------------------------------
# Find OLD feature file
# ------------------------------------------------------------

OLD_FEATURES = None
for fp in OLD_FEATURE_CANDIDATES:
    if fp.exists():
        OLD_FEATURES = fp
        break

if OLD_FEATURES is None:
    raise FileNotFoundError("No OLD feature file found.")

print("Using OLD feature file:")
print(OLD_FEATURES)

# ------------------------------------------------------------
# 1. Check OLD feature 2025 rows
# ------------------------------------------------------------

feat = pd.read_csv(OLD_FEATURES, low_memory=False)
feat["time"] = pd.to_datetime(feat["time"], errors="coerce", utc=True).dt.tz_convert(None)

feat25 = feat[feat["time"].dt.year == 2025].copy()

print("\n" + "=" * 70)
print("OLD FEATURES — 2025")
print("=" * 70)
print("Total feature rows:", len(feat))
print("2025 feature rows:", len(feat25))
print("2025 time range:", feat25["time"].min(), "to", feat25["time"].max())

print("\n2025 feature NaN fractions:")
print(feat25[[
    "eta850_abs", "rh600", "wshear_200_850",
    "sst_K", "mslp_Pa", "pi_vmax_ms", "gpi",
    "speed_kmh", "direction_deg"
]].isna().mean())

# ------------------------------------------------------------
# 2. Check OLD updated labels have 2025 positives
# ------------------------------------------------------------

if not OLD_LABEL_UPDATED.exists():
    raise FileNotFoundError(OLD_LABEL_UPDATED)

lab = pd.read_csv(OLD_LABEL_UPDATED, low_memory=False)
lab["time"] = pd.to_datetime(lab["time"], errors="coerce", utc=True).dt.tz_convert(None)

lab25 = lab[lab["time"].dt.year == 2025].copy()

print("\n" + "=" * 70)
print("OLD UPDATED LABELS — 2025")
print("=" * 70)
print("Total label rows:", len(lab))
print("2025 label rows:", len(lab25))

for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg={int(lab25[f'box2deg_y_{L}h'].sum())} | "
        f"radius300km={int(lab25[f'radius300km_y_{L}h'].sum())}"
    )

# ------------------------------------------------------------
# 3. Check OLD updated master has 2025 positives
# ------------------------------------------------------------

if not OLD_MASTER_UPDATED.exists():
    raise FileNotFoundError(OLD_MASTER_UPDATED)

master = pd.read_csv(OLD_MASTER_UPDATED, low_memory=False)
master["time"] = pd.to_datetime(master["time"], errors="coerce", utc=True).dt.tz_convert(None)

m25 = master[master["time"].dt.year == 2025].copy()

print("\n" + "=" * 70)
print("OLD UPDATED MASTER — 2025")
print("=" * 70)
print("Total master rows:", len(master))
print("2025 master rows:", len(m25))

for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg={int(m25[f'box2deg_y_{L}h'].sum())} | "
        f"radius300km={int(m25[f'radius300km_y_{L}h'].sum())}"
    )

# ------------------------------------------------------------
# 4. Check OLD updated case tables include 2025 rows/positives
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OLD UPDATED CASE TABLES — 2025")
print("=" * 70)

for L in [6, 12, 24, 48]:
    fp = OLD_CASE_DIR / f"box2deg_case_table_old_{L}h.csv"

    if not fp.exists():
        print(f"\n{L}h missing:", fp)
        continue

    case = pd.read_csv(fp, low_memory=False)
    case["time"] = pd.to_datetime(case["time"], errors="coerce", utc=True).dt.tz_convert(None)

    c25 = case[case["time"].dt.year == 2025].copy()

    print(f"\nLead {L}h")
    print("case rows:", len(case))
    print("case positives:", int(case["y"].sum()))
    print("2025 case rows:", len(c25))
    print("2025 positives:", int(c25["y"].sum()))
    print("2025 negatives:", int((c25["y"] == 0).sum()))

Using OLD feature file:
C:\Users\_s2218026\Documents\lab\mcs_output\mcs_features_PI_GPI_old_2009_2025_JJASO_zhang4pt.csv

OLD FEATURES — 2025
Total feature rows: 900585
2025 feature rows: 60874
2025 time range: 2025-06-01 00:00:00 to 2025-10-31 21:00:00

2025 feature NaN fractions:
eta850_abs        1.000000
rh600             1.000000
wshear_200_850    1.000000
sst_K             1.000000
mslp_Pa           1.000000
pi_vmax_ms        1.000000
gpi               1.000000
speed_kmh         0.447055
direction_deg     0.447055
dtype: float64

OLD UPDATED LABELS — 2025
Total label rows: 900585
2025 label rows: 60874
6h | box2deg=21 | radius300km=25
12h | box2deg=21 | radius300km=29
24h | box2deg=14 | radius300km=20
48h | box2deg=9 | radius300km=17

OLD UPDATED MASTER — 2025
Total master rows: 900585
2025 master rows: 60874
6h | box2deg=21 | radius300km=25
12h | box2deg=21 | radius300km=29
24h | box2deg=14 | radius300km=20
48h | box2deg=9 | radius300km=17

OLD UPDATED CASE TABLES — 2025

Lead 6